# Advanced Model Improvements & Optimization
## Citizen Grievance Analysis System

This notebook focuses on **improving model performance** through:

### 🎯 Key Improvements:

1. **Hyperparameter Tuning** - Grid Search and Random Search
2. **Feature Engineering** - Advanced text features
3. **Ensemble Methods** - Voting and Stacking Classifiers
4. **Model Explainability** - SHAP and LIME
5. **Class Imbalance Handling** - SMOTE and class weights
6. **Feature Importance Analysis**
7. **Error Analysis** - Understand misclassifications
8. **Model Calibration** - Probability calibration
9. **Advanced Evaluation** - ROC curves, PR curves
10. **Production Optimization** - Model compression and speed improvements

---

## 1. Setup and Import Libraries

In [ ]:
# Install additional packages for improvements
!pip install scikit-learn pandas numpy matplotlib seaborn shap lime imbalanced-learn optuna joblib

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import time
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score,
    StratifiedKFold
)
from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import label_binarize

# Imbalanced learning
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Explainability
import shap
from lime.lime_text import LimeTextExplainer

# Hyperparameter optimization
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ All libraries imported successfully!")
print(f"📅 Notebook executed on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Load Trained Models and Data

Load the models and data from the previous training notebook.

In [ ]:
# Load saved models
print("Loading trained models and artifacts...")

try:
    # Load models
    nb_model = joblib.load('trained_models/naive_bayes.pkl')
    lr_model = joblib.load('trained_models/logistic_regression.pkl')
    svm_model = joblib.load('trained_models/linear_svm.pkl')
    rf_model = joblib.load('trained_models/random_forest.pkl')
    
    # Load vectorizer
    tfidf_vectorizer = joblib.load('trained_models/tfidf_vectorizer.pkl')
    
    # Load label encoder
    label_encoder = joblib.load('trained_models/label_encoder.pkl')
    
    # Load results
    results_df = pd.read_csv('trained_models/model_comparison_results.csv')
    
    print("✅ All models loaded successfully!")
    print(f"\nLoaded models:")
    print("  - Naive Bayes")
    print("  - Logistic Regression")
    print("  - Linear SVM")
    print("  - Random Forest")
    print("\nLoaded artifacts:")
    print("  - TF-IDF Vectorizer")
    print("  - Label Encoder")
    print("  - Previous Results")
    
except FileNotFoundError:
    print("⚠️  Warning: Trained models not found!")
    print("Please run the 'model_training_evaluation.ipynb' notebook first.")
    print("\nYou can also load your dataset here and train from scratch.")

In [ ]:
# Note: You'll need to load your actual dataset here
# For this notebook to work, you need X_train, X_test, y_train, y_test
# from your original dataset

print("\n⚠️  IMPORTANT: Load your dataset here")
print("="*80)
print("This notebook requires the original train/test split.")
print("\nEither:")
print("1. Re-run the data preprocessing from model_training_evaluation.ipynb")
print("2. Or load pre-saved train/test data")
print("\nFor demonstration, we'll create placeholder variables.")
print("Replace these with your actual data!")
print("="*80)

## 3. Hyperparameter Tuning

Optimize model performance through systematic hyperparameter search.

### 3.1 Grid Search for Logistic Regression

In [ ]:
# Define parameter grid
param_grid_lr = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'solver': ['saga', 'liblinear'],
    'max_iter': [500, 1000]
}

print("Starting Grid Search for Logistic Regression...")
print(f"Parameter grid: {param_grid_lr}")
print(f"Total combinations: {np.prod([len(v) for v in param_grid_lr.values()])}")

# Note: Uncomment and run with your actual data
'''
from sklearn.linear_model import LogisticRegression

grid_search_lr = GridSearchCV(
    LogisticRegression(random_state=RANDOM_SEED),
    param_grid_lr,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)

grid_search_lr.fit(X_train, y_train)

print(f"\n✅ Grid Search Complete!")
print(f"Best parameters: {grid_search_lr.best_params_}")
print(f"Best CV score: {grid_search_lr.best_score_:.4f}")

# Evaluate on test set
y_pred_tuned = grid_search_lr.predict(X_test)
print(f"Test F1 score: {f1_score(y_test, y_pred_tuned, average='weighted'):.4f}")
'''

print("\n💡 Grid Search cell prepared. Uncomment to run with your data.")

### 3.2 Random Search for Random Forest

In [ ]:
# Define parameter distribution
param_dist_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

print("Starting Random Search for Random Forest...")
print(f"Parameter distribution: {param_dist_rf}")

# Note: Uncomment and run with your actual data
'''
from sklearn.ensemble import RandomForestClassifier

random_search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED),
    param_dist_rf,
    n_iter=20,  # Number of random combinations to try
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_SEED
)

random_search_rf.fit(X_train, y_train)

print(f"\n✅ Random Search Complete!")
print(f"Best parameters: {random_search_rf.best_params_}")
print(f"Best CV score: {random_search_rf.best_score_:.4f}")
'''

print("\n💡 Random Search cell prepared. Uncomment to run with your data.")

### 3.3 Optuna Optimization (Advanced)

In [ ]:
# Optuna for advanced hyperparameter optimization
print("Optuna optimization example:")
print("Optuna is a powerful framework for hyperparameter optimization.")
print("It uses advanced algorithms like TPE (Tree-structured Parzen Estimator).")

# Example objective function
'''
def objective(trial):
    # Suggest hyperparameters
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
    }
    
    # Train model
    model = RandomForestClassifier(**params, random_state=RANDOM_SEED)
    
    # Cross-validation
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted')
    
    return scores.mean()

# Create study
study = optuna.create_study(direction='maximize', study_name='rf_optimization')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✅ Optimization Complete!")
print(f"Best parameters: {study.best_params}")
print(f"Best value: {study.best_value:.4f}")

# Visualize optimization
plot_optimization_history(study)
plt.show()
'''

print("\n💡 Optuna optimization template prepared.")

## 4. Ensemble Methods

Combine multiple models for improved performance.

### 4.1 Voting Classifier

In [ ]:
# Create voting ensemble
print("Creating Voting Classifier...")

# Note: Uncomment and run with your actual models
'''
voting_clf = VotingClassifier(
    estimators=[
        ('nb', nb_model),
        ('lr', lr_model),
        ('rf', rf_model)
    ],
    voting='soft'  # Use predicted probabilities
)

print("Training Voting Classifier...")
voting_clf.fit(X_train, y_train)

# Evaluate
y_pred_voting = voting_clf.predict(X_test)
accuracy_voting = accuracy_score(y_test, y_pred_voting)
f1_voting = f1_score(y_test, y_pred_voting, average='weighted')

print(f"\n✅ Voting Classifier Results:")
print(f"Accuracy: {accuracy_voting:.4f}")
print(f"F1 Score: {f1_voting:.4f}")
'''

print("\n💡 Voting classifier combines predictions from multiple models.")
print("   - Hard voting: Majority vote")
print("   - Soft voting: Average predicted probabilities")

### 4.2 Stacking Classifier

In [ ]:
# Create stacking ensemble
print("Creating Stacking Classifier...")

# Note: Uncomment and run with your actual models
'''
from sklearn.linear_model import LogisticRegression

stacking_clf = StackingClassifier(
    estimators=[
        ('nb', nb_model),
        ('lr', lr_model),
        ('rf', rf_model)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

print("Training Stacking Classifier...")
stacking_clf.fit(X_train, y_train)

# Evaluate
y_pred_stacking = stacking_clf.predict(X_test)
accuracy_stacking = accuracy_score(y_test, y_pred_stacking)
f1_stacking = f1_score(y_test, y_pred_stacking, average='weighted')

print(f"\n✅ Stacking Classifier Results:")
print(f"Accuracy: {accuracy_stacking:.4f}")
print(f"F1 Score: {f1_stacking:.4f}")
'''

print("\n💡 Stacking uses a meta-learner to combine base model predictions.")

## 5. Handling Class Imbalance

Improve performance on minority classes.

### 5.1 SMOTE (Synthetic Minority Over-sampling)

In [ ]:
# Apply SMOTE
print("Applying SMOTE for class balancing...")

# Note: Uncomment and run with your actual data
'''
smote = SMOTE(random_state=RANDOM_SEED)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\n✅ SMOTE Applied!")
print(f"Original training size: {X_train.shape[0]:,}")
print(f"Balanced training size: {X_train_balanced.shape[0]:,}")

print(f"\nClass distribution before SMOTE:")
print(pd.Series(y_train).value_counts())

print(f"\nClass distribution after SMOTE:")
print(pd.Series(y_train_balanced).value_counts())

# Train model on balanced data
lr_balanced = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
lr_balanced.fit(X_train_balanced, y_train_balanced)

# Evaluate
y_pred_balanced = lr_balanced.predict(X_test)
print(f"\nF1 Score with SMOTE: {f1_score(y_test, y_pred_balanced, average='weighted'):.4f}")
'''

print("\n💡 SMOTE creates synthetic examples of minority classes.")

### 5.2 Class Weight Adjustment

In [ ]:
# Use class weights
print("Training with balanced class weights...")

# Note: Uncomment and run with your actual data
'''
from sklearn.linear_model import LogisticRegression

lr_weighted = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Automatically adjust weights
    random_state=RANDOM_SEED
)

lr_weighted.fit(X_train, y_train)
y_pred_weighted = lr_weighted.predict(X_test)

print(f"\n✅ Model trained with balanced class weights")
print(f"F1 Score: {f1_score(y_test, y_pred_weighted, average='weighted'):.4f}")
print(f"Macro F1: {f1_score(y_test, y_pred_weighted, average='macro'):.4f}")
'''

print("\n💡 Class weights penalize errors on minority classes more heavily.")

## 6. Model Explainability with SHAP

Understand which features drive model predictions.

In [ ]:
# SHAP Explainer
print("Creating SHAP explainer...")

# Note: Uncomment and run with your actual model and data
'''
# Create explainer
explainer = shap.LinearExplainer(lr_model, X_train[:100])  # Use sample for speed

# Calculate SHAP values
shap_values = explainer.shap_values(X_test[:100])

# Summary plot
print("\nGenerating SHAP summary plot...")
shap.summary_plot(shap_values, X_test[:100], 
                  feature_names=tfidf_vectorizer.get_feature_names_out())

# Force plot for single prediction
print("\nGenerating SHAP force plot for single prediction...")
shap.force_plot(explainer.expected_value[0], 
                shap_values[0][0], 
                X_test[0],
                feature_names=tfidf_vectorizer.get_feature_names_out())
'''

print("\n💡 SHAP (SHapley Additive exPlanations) provides feature importance.")
print("   - Shows which words/features influenced each prediction")
print("   - Helps build trust in model decisions")

## 7. LIME Explanations

Local interpretable model-agnostic explanations.

In [ ]:
# LIME Text Explainer
print("Creating LIME explainer for text...")

# Note: Uncomment and run with your actual data
'''
# Create explainer
lime_explainer = LimeTextExplainer(class_names=label_encoder.classes_)

# Define prediction function
def predict_proba_text(texts):
    vectors = tfidf_vectorizer.transform(texts)
    return lr_model.predict_proba(vectors)

# Explain a single prediction
sample_text = X_test_text[0]  # You need the original text, not TF-IDF
explanation = lime_explainer.explain_instance(
    sample_text,
    predict_proba_text,
    num_features=10
)

print("\nLIME Explanation:")
explanation.show_in_notebook(text=True)

# HTML visualization
explanation.save_to_file('lime_explanation.html')
print("\n✅ LIME explanation saved to 'lime_explanation.html'")
'''

print("\n💡 LIME explains individual predictions by approximating the model locally.")

## 8. Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
print("Analyzing feature importance...")

# Note: Uncomment and run with your actual model
'''
# Get feature importances
feature_importances = rf_model.feature_importances_
feature_names = tfidf_vectorizer.get_feature_names_out()

# Create dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(12, 8))
top_20 = importance_df.head(20)
plt.barh(range(len(top_20)), top_20['importance'])
plt.yticks(range(len(top_20)), top_20['feature'])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 20 Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\n✅ Top 10 most important features:")
print(importance_df.head(10).to_string(index=False))
'''

print("\n💡 Feature importance shows which words are most predictive.")

## 9. Error Analysis

Understand where the model makes mistakes.

In [ ]:
# Error analysis
print("Performing error analysis...")

# Note: Uncomment and run with your actual predictions
'''
# Identify misclassifications
errors = y_test != y_pred
error_indices = np.where(errors)[0]

print(f"\n📊 Error Statistics:")
print(f"Total test samples: {len(y_test)}")
print(f"Misclassifications: {errors.sum()}")
print(f"Error rate: {errors.mean():.2%}")

# Analyze errors by class
error_df = pd.DataFrame({
    'true_label': y_test[errors],
    'predicted_label': y_pred[errors]
})

print(f"\n🔍 Most Common Misclassifications:")
print(error_df.value_counts().head(10))

# Examine specific errors
print(f"\n📝 Sample Misclassifications:")
for idx in error_indices[:5]:
    print(f"\nTrue: {y_test[idx]} | Predicted: {y_pred[idx]}")
    print(f"Text: {X_test_text[idx][:200]}...")
    print("-" * 80)
'''

print("\n💡 Error analysis helps identify patterns in misclassifications.")

## 10. Advanced Evaluation Metrics

### 10.1 ROC Curves and AUC

In [ ]:
# ROC curves for multi-class
print("Generating ROC curves...")

# Note: Uncomment and run with your actual data
'''
# Binarize the labels
y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_bin.shape[1]

# Get predicted probabilities
y_score = lr_model.predict_proba(X_test)

# Compute ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(12, 8))
colors = plt.cm.rainbow(np.linspace(0, 1, n_classes))

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'{label_encoder.classes_[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Multi-class Classification', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
'''

print("\n💡 ROC curves show the trade-off between true and false positive rates.")

### 10.2 Precision-Recall Curves

In [ ]:
# Precision-Recall curves
print("Generating Precision-Recall curves...")

# Note: Uncomment and run with your actual data
'''
# Compute PR curve for each class
precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
    average_precision[i] = average_precision_score(y_test_bin[:, i], y_score[:, i])

# Plot PR curves
plt.figure(figsize=(12, 8))

for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label=f'{label_encoder.classes_[i]} (AP = {average_precision[i]:.2f})')

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves - Multi-class', fontsize=14, fontweight='bold')
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
'''

print("\n💡 PR curves are especially useful for imbalanced datasets.")

## 11. Model Calibration

Ensure predicted probabilities are reliable.

In [ ]:
# Calibrate model
print("Calibrating model probabilities...")

# Note: Uncomment and run with your actual model
'''
# Calibrate using Platt scaling
calibrated_clf = CalibratedClassifierCV(lr_model, method='sigmoid', cv=5)
calibrated_clf.fit(X_train, y_train)

# Get calibrated probabilities
y_prob_uncalibrated = lr_model.predict_proba(X_test)
y_prob_calibrated = calibrated_clf.predict_proba(X_test)

# Plot calibration curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# For each class (showing first 3 for clarity)
for idx, class_idx in enumerate(range(min(3, n_classes))):
    # Uncalibrated
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test_bin[:, class_idx], y_prob_uncalibrated[:, class_idx], n_bins=10
    )
    axes[0].plot(mean_predicted_value, fraction_of_positives, 's-', 
                 label=f'{label_encoder.classes_[class_idx]}')
    
    # Calibrated
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test_bin[:, class_idx], y_prob_calibrated[:, class_idx], n_bins=10
    )
    axes[1].plot(mean_predicted_value, fraction_of_positives, 's-',
                 label=f'{label_encoder.classes_[class_idx]}')

# Perfect calibration line
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')

axes[0].set_title('Before Calibration', fontsize=14, fontweight='bold')
axes[1].set_title('After Calibration', fontsize=14, fontweight='bold')

for ax in axes:
    ax.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax.set_ylabel('Fraction of Positives', fontsize=12)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Model calibrated!")
'''

print("\n💡 Calibration ensures predicted probabilities match actual frequencies.")

## 12. Production Optimizations

### 12.1 Model Size Reduction

In [ ]:
# Reduce model size
print("Optimizing model for production...")

# Technique 1: Reduce number of features
print("\n1️⃣ Feature reduction:")
print("   - Use fewer TF-IDF features (e.g., max_features=2000 instead of 5000)")
print("   - Select only most important features")

# Technique 2: Model compression
print("\n2️⃣ Model compression:")
print("   - Use simpler models (e.g., Logistic Regression instead of Random Forest)")
print("   - Prune decision trees")

# Technique 3: Quantization
print("\n3️⃣ Quantization:")
print("   - Convert float32 to float16")
print("   - Use integer quantization for deep learning models")

# Example: Check model sizes
'''
import sys

print("\n📊 Model Sizes:")
print(f"TF-IDF Vectorizer: {sys.getsizeof(joblib.dumps(tfidf_vectorizer)) / 1024 / 1024:.2f} MB")
print(f"Logistic Regression: {sys.getsizeof(joblib.dumps(lr_model)) / 1024 / 1024:.2f} MB")
print(f"Random Forest: {sys.getsizeof(joblib.dumps(rf_model)) / 1024 / 1024:.2f} MB")
'''

print("\n💡 Smaller models = faster inference + lower memory usage.")

### 12.2 Inference Speed Optimization

In [ ]:
# Measure inference speed
print("Measuring inference speed...")

# Note: Uncomment and run with your actual models
'''
import time

models = {
    'Naive Bayes': nb_model,
    'Logistic Regression': lr_model,
    'Linear SVM': svm_model,
    'Random Forest': rf_model
}

inference_times = {}

for name, model in models.items():
    start = time.time()
    _ = model.predict(X_test)
    elapsed = time.time() - start
    inference_times[name] = elapsed
    print(f"{name:20s}: {elapsed:.4f}s ({X_test.shape[0]/elapsed:.0f} predictions/sec)")

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(list(inference_times.keys()), list(inference_times.values()), color='skyblue')
plt.xlabel('Inference Time (seconds)', fontsize=12)
plt.title('Model Inference Speed Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
'''

print("\n💡 Choose faster models for real-time applications.")

## 13. Summary and Recommendations

In [ ]:
# Generate improvement summary
summary = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║                 MODEL IMPROVEMENT TECHNIQUES SUMMARY                          ║
╚═══════════════════════════════════════════════════════════════════════════════╝

✅ TECHNIQUES COVERED:

1. Hyperparameter Tuning
   - Grid Search for exhaustive search
   - Random Search for efficient exploration
   - Optuna for advanced optimization

2. Ensemble Methods
   - Voting Classifier (hard/soft voting)
   - Stacking with meta-learner
   - Combines strengths of multiple models

3. Class Imbalance Handling
   - SMOTE for synthetic oversampling
   - Class weight adjustment
   - Improved minority class performance

4. Model Explainability
   - SHAP for global feature importance
   - LIME for local explanations
   - Build trust and understanding

5. Feature Analysis
   - Feature importance from tree models
   - Identify most predictive words

6. Error Analysis
   - Identify common misclassifications
   - Target improvements to specific cases

7. Advanced Metrics
   - ROC curves and AUC
   - Precision-Recall curves
   - Better evaluation for imbalanced data

8. Model Calibration
   - Reliable probability predictions
   - Important for confidence scores

9. Production Optimization
   - Model size reduction
   - Inference speed optimization
   - Ready for deployment

╔═══════════════════════════════════════════════════════════════════════════════╗
║                          RECOMMENDATIONS                                      ║
╚═══════════════════════════════════════════════════════════════════════════════╝

📈 FOR BEST PERFORMANCE:
   1. Start with hyperparameter tuning on best base model
   2. Try ensemble methods for additional boost
   3. Handle class imbalance if minority classes underperform

🔍 FOR INTERPRETABILITY:
   1. Use SHAP for overall feature importance
   2. Use LIME to explain individual predictions
   3. Show feature importance to stakeholders

🚀 FOR PRODUCTION:
   1. Choose faster models (Logistic Regression, Linear SVM)
   2. Reduce feature dimensions if needed
   3. Calibrate probabilities for confidence scores
   4. Monitor performance on new data

💡 KEY INSIGHT:
   Different techniques serve different purposes. Choose based on:
   - Performance requirements
   - Interpretability needs
   - Production constraints
   - Available computational resources

╔═══════════════════════════════════════════════════════════════════════════════╗
║                        NEXT STEPS                                             ║
╚═══════════════════════════════════════════════════════════════════════════════╝

1. Load your actual trained models and data
2. Uncomment and run the improvement techniques
3. Compare results with baseline models
4. Select best approach for your use case
5. Save improved models for deployment

"""

print(summary)

# Save summary
with open('improvement_summary.txt', 'w') as f:
    f.write(summary)

print("\n✅ Summary saved to 'improvement_summary.txt'")

## 📚 Conclusion

This notebook provides a comprehensive toolkit for improving machine learning models. Each technique addresses different aspects:

- **Performance**: Hyperparameter tuning, ensembles, class balancing
- **Understanding**: SHAP, LIME, feature importance, error analysis
- **Reliability**: Calibration, advanced metrics
- **Deployment**: Optimization, speed, size reduction

### 🎯 Usage Instructions:

1. **Load your data and models** from the training notebook
2. **Uncomment the code cells** you want to run
3. **Experiment** with different techniques
4. **Compare results** to find the best approach
5. **Document improvements** for your project

### ⚡ Pro Tips:

- Start with quick wins (class weights, better hyperparameters)
- Use cross-validation to verify improvements
- Don't overfit to the test set!
- Consider the cost-benefit of each technique

---

**Ready for production!** 🚀